# Сохранение полного датасета с рекомендациями

В рамках проекта сохранялись рекомендации только для тестовых пользователей, для системы рекомендаций необходимо иметь:
1. рекомендации для всех "горячих" пользователей; 
2. похожие треки для всех треков и желательно получить достаточно большой список;
3. последние треки пользователей, в том числе прослушанные "теплыми" пользователями.

## Инициализация

In [1]:
import os
import pandas as pd
import numpy as np
import joblib
import scipy
from dotenv import load_dotenv
from catboost import CatBoostClassifier

# Определение констант
load_dotenv()

RANDOM_STATE = 42

# Дата для разбиения выборки на тренировочную и тестовую
SPLIT_DATE = pd.to_datetime("2022-12-16")

S3_BUCKET = os.getenv("AWS_S3_BUCKET")

STORAGE_OPTIONS={
    "endpoint_url": os.getenv("AWS_ENDPOINT_URL"), 
    "key": os.getenv("AWS_ACCESS_KEY_ID"), 
    "secret": os.getenv("AWS_SECRET_ACCESS_KEY"),
    "client_kwargs":{"region_name": os.getenv("AWS_REGION")},
    "config_kwargs": {"signature_version": "s3v4"}
}

S3 = True

In [2]:
# Функция загрузки данных

def load_data(
        file_name: str, dir_name: str, 
        columns = None, filters = None,
        s3: bool = S3, bucket = S3_BUCKET, 
        storage_options = STORAGE_OPTIONS) -> pd.DataFrame:
    """
    Функция для загрузки датасета pandas по указанному имени файла `file_name` 
    из указанной директории `dir_name`. Параметр `s3` регулирует загрузку локально или 
    из директории 'recsys' хранилища S3 указанного `bucket` с применением `storage_options`.
    Параметр `columns` - список загружаемых столбцов.
    Параметр `filters` - фильтр вида `[('user_id', 'in', var), ...]`
    """
    if s3:
        return pd.read_parquet(
            f"s3://{bucket}/recsys/{dir_name}/{file_name}",
            engine="pyarrow",
            storage_options=storage_options, 
            columns=columns, filters=filters
        )
    else:
        return pd.read_parquet(
            f"../{dir_name}/{file_name}", 
            engine="pyarrow",
            columns=columns, filters=filters
        )
    
# Функция сохранения данных
 
def save_data(df: pd.DataFrame, file_name: str, dir_name: str, index = False, 
              s3: bool = S3, bucket = S3_BUCKET, storage_options = STORAGE_OPTIONS):
    """
    Функция для сохранения датасета pandas по указанному имени файла `file_name` 
    в указанной директории `dir_name`. Параметр `s3` регулирует сохранение локально или 
    в хранилище S3 в директорию 'recsys' указанного `bucket` с применением `storage_options`.
    """
    if s3:
        df.to_parquet(
            f"s3://{bucket}/recsys/{dir_name}/{file_name}", 
            engine="pyarrow", index=index,
            storage_options=storage_options
        )
    else:
        df.to_parquet(f"../{dir_name}/{file_name}", index=True)

# Функция извлечения таргетов и новой тестовой выборки 
def split_target_test(df: pd.DataFrame, hot_user_ids: list):
    """
    Функция разбиения тестовой выборки на таргет и новый тест.

    Parameters
    ----------
    df : DataFrame
        Тестовая выборка для разбиения
    hot_user_ids : list
        Список id горячих пользователей

    Returns
    -------
    (target, test) : (DataFrame, Dataframe)
        Возвращает выборку для таргетов и новую тестовую выборку
    """
    # Ранжирование треков
    target_ids = df.groupby("user_id")["item_seq"].rank(
        method="dense", ascending=True)
    # Определение ранних треков
    target_ids = target_ids <= np.ceil(
        df.groupby("user_id")["item_id"].transform("count") / 2)
    # Получение горячих пользователей
    target_ids = (
        target_ids &
        df.reset_index().merge(
            pd.DataFrame({"user_id": hot_user_ids, "hot": True}),
            how="left", on="user_id"
        ).set_index("index")["hot"].notna()
    )
    return df[target_ids], df[~target_ids]

# Обобщенная логистическая функция (обратная)
def asym_smoothing(value, min=0, q=0.01, a=5, b=5, c=1/3):
    # ограничиваем сверху для экспоненты
    value = np.clip(value, None, 3500)
    return (
        min + (1 - min) / ((1 + q * np.exp(-(a - value) / b)) ** c)
    ).astype(np.float32)

In [3]:
# Загрузка энкодеров id
with open("../models/id_encoders.pkl", "rb") as fd:
    encoders = joblib.load(fd)

In [4]:
# Загрузка модели ALS
with open("../models/als_model.pkl", "rb") as fd:
    als_model = joblib.load(fd)

In [5]:
# Загрузка событий до разбиения данных
cols = ["user_id", "item_id", "item_seq"]
train = load_data(
    file_name="events.parquet", dir_name="data", 
    columns=cols, filters=[("started_at", "<", SPLIT_DATE)]
)

# Загрузка и разделение на таргет и тест
test = load_data(
    "events.parquet", "data", columns=cols,
    filters=[("started_at", ">=", SPLIT_DATE)]
)

# Разделение на таргет и тест
target, test = split_target_test(
    test, 
    encoders["user"].classes_)

# Добавление событий из таргета
train = pd.concat([train, target[cols]], ignore_index=True)

del target, test

## Последние треки пользователей

In [6]:
# Обратное ранжирование событий
train["item_seq"] = train.groupby("user_id")["item_seq"] \
    .transform("max") - train["item_seq"]

# Получение последних событий пользователей
last_events = train[train["item_seq"] < 5] \
    .sort_values(["user_id", "item_seq"]) 
last_events

,user_id,item_id,item_seq
25,0,20497621,0
24,0,20232119,1
23,0,18102829,2
22,0,18042047,3
21,0,17802674,4
...,...,...,...
205765177,1374582,85483999,0
205765176,1374582,85347315,1
205765175,1374582,84823577,2
205765174,1374582,84657263,3


In [7]:
# Получение списков последних треков
last_events = last_events.groupby("user_id")["item_id"].agg(list).reset_index()
last_events

,user_id,item_id
0,0,"[20497621, 20232119, 18102829, 18042047, 17802..."
1,1,"[83436771, 80631189, 80154987, 78877209, 78877..."
2,2,"[71650200, 65460089, 65457108, 60207692, 35946..."
3,3,"[78194999, 68348391, 68348390, 68348389, 68348..."
4,4,"[86153070, 84487963, 84099295, 83858540, 83764..."
...,...,...
1342561,1374578,"[35279525, 34976783, 34726523, 34686730, 33977..."
1342562,1374579,"[41899516, 26491282, 24327488, 18042047, 17902..."
1342563,1374580,"[96618101, 94848971, 93126179, 88023664, 87813..."
1342564,1374581,"[95834048, 95070082, 94739018, 94696082, 94303..."


In [34]:
# Сохранение датасета
save_data(
    last_events, "last_events.parquet", 
    "recommendations")
del cols, last_events

In [14]:
from sklearn.preprocessing import LabelEncoder
encoders["warm"] = LabelEncoder()
encoders["warm"].fit(
    train.groupby("user_id")["item_id"].count() \
        .loc[lambda x: x < 5] \
        .index.to_list()
)
with open("../models/id_encoders.pkl", "wb") as fd:
    joblib.dump(encoders, fd)

## Похожие треки

In [ ]:
# Получение предсказаний
similar = als_model.similar_items(
    range(len(encoders["item"].classes_)),N=51)

In [7]:
# преобразуем полученные рекомендации в табличный формат
ids_enc = similar[0] 
scores = similar[1]

similar = pd.DataFrame({
    "item_id": range(len(encoders["item"].classes_)),
    "similar_id": ids_enc.tolist(), 
    "score": scores.tolist()})
similar

,item_id,similar_id,score
0,0,"[0, 343382, 438221, 194445, 24547, 428075, 309...","[0.9999999403953552, 0.9639434814453125, 0.951..."
1,1,"[1, 299413, 240798, 240793, 358526, 262237, 30...","[1.0, 0.9639357328414917, 0.9558799266815186, ..."
2,2,"[2, 289368, 21542, 275126, 373069, 419368, 237...","[1.0000001192092896, 0.9397692680358887, 0.939..."
3,3,"[3, 5, 10, 8, 333260, 20021, 432728, 410431, 4...","[1.0, 0.9999997019767761, 0.999999463558197, 0..."
4,4,"[4, 208313, 159468, 210698, 168201, 197973, 10...","[0.9999998807907104, 0.972038984298706, 0.9659..."
...,...,...,...
999847,999847,"[999847, 988771, 990032, 999370, 993718, 98909...","[1.0000001192092896, 0.9871659874916077, 0.984..."
999848,999848,"[999848, 992993, 979742, 986728, 999386, 98866...","[1.000000238418579, 0.9652305245399475, 0.9595..."
999849,999849,"[999849, 992589, 991476, 994703, 991244, 99623...","[1.0000001192092896, 0.9730508923530579, 0.970..."
999850,999850,"[999850, 999074, 997736, 998425, 999593, 99236...","[0.9999997615814209, 0.977697491645813, 0.9677..."


In [8]:
similar.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 999852 entries, 0 to 999851
Data columns (total 3 columns):
 #   Column      Non-Null Count   Dtype 
---  ------      --------------   ----- 
 0   item_id     999852 non-null  int64 
 1   similar_id  999852 non-null  object
 2   score       999852 non-null  object
dtypes: int64(1), object(2)
memory usage: 22.9+ MB


In [9]:
# преобразование списков 'similar_id' и 'score' в строки, 
# с отбором только неодинаковых треков
similar = similar.explode(["similar_id", "score"], ignore_index=True) \
    .loc[lambda x: x["item_id"] != x["similar_id"]]

In [10]:
# приведение типов данных
similar["similar_id"] = similar["similar_id"].astype(np.int32)
similar["score"] = similar["score"].astype(np.float32)

In [13]:
# получеине изначальных идентификаторов
similar["item_id"] = encoders["item"].inverse_transform(similar["item_id"])
similar["similar_id"] = encoders["item"].inverse_transform(similar["similar_id"])
similar

,item_id,similar_id,score
1,26,22889677,0.963943
2,26,30373026,0.951161
3,26,6493239,0.944056
4,26,152306,0.943156
5,26,29632247,0.942152
...,...,...,...
20996887,101521819,96072309,0.924861
20996888,101521819,97770279,0.923609
20996889,101521819,101490148,0.923253
20996890,101521819,95052946,0.922784


In [14]:
# Сохранение датасета
save_data(similar, "similar.parquet", "recommendations")

del similar

## Итоговые рекомендации

In [ ]:
# Удаление теплых пользователей
train = train[
    train.groupby("user_id")["item_id"].transform("count") >= 5
].copy()

In [ ]:
# Преобразование id пользователей и объектов
for col in ["user", "item"]:
    train[f"{col}_id"] = encoders[col].transform(train[f"{col}_id"]).astype(np.int32)

# Добавление весов
train["weight"] = asym_smoothing(train["item_seq"])

In [ ]:
# Подготовка разреженной матрицы
train_sm = scipy.sparse.csr_matrix(
    (train["weight"], (train["user_id"], train["item_id"]))
)

del train

In [ ]:
# Получение рекомендаций ALS
user_ids_enc = range(len(encoders["user"].classes_))
personal_als = als_model.recommend(
    user_ids_enc, train_sm[user_ids_enc], N=100)
del train_sm

In [ ]:
# Преобразование рекомендаций в табличный формат
item_ids_enc = personal_als[0] # type: ignore
als_scores = personal_als[1] # type: ignore

personal_als = pd.DataFrame({
    "user_id": user_ids_enc,
    "item_id": item_ids_enc.tolist(), 
    "score": als_scores.tolist()})

# Получение корректных id пользователя
personal_als["user_id"] = encoders["user"].inverse_transform(
    personal_als["user_id"])

# Преобразование списков 'item_id' и 'score' в строки
personal_als = personal_als.explode(["item_id", "score"], ignore_index=True)

# Приведение типов данных
personal_als["item_id"] = personal_als["item_id"].astype(np.int32)
personal_als["score"] = personal_als["score"].astype(np.float32)

# Получение корректных id треков
personal_als["item_id"] = encoders["item"].inverse_transform(
    personal_als["item_id"])

personal_als

In [ ]:
# Сохранение датасета
save_data(personal_als, "personal_als.parquet", "recommendations")

In [ ]:
# Загрузка при небходимости
personal_als = load_data("personal_als.parquet", "recommendations")

In [ ]:
# Загрузка дополнительных данных для ранжирования
item_pop = load_data("top_popular.parquet", "recommendations")
item_feats = load_data("item_feats.parquet", "features")
user_feats = load_data("hot_user_feats.parquet", "features")

In [ ]:
# Функция для получения датасета с признаками для ранжирующей модели
def add_features(
        df_als: pd.DataFrame,
        item_pop = item_pop,
        item_feats = item_feats,
        user_feats = user_feats
):
    """
    Функция для подготовки признаков для ранжирующей модели 
    путем добавления к персональным рекомендациям признаков 
    из сохраненных датасетов и удаления идентификаторов 
    пользователей и треков
    """
    # Работа с копией датасета
    df_als = df_als.copy()
    
    # Добавление популярности треков
    df_als = df_als.merge(
        item_pop, how="left", on="item_id")

    # Добавление признаков трека
    df_als = df_als.merge(
        item_feats,how="left", on="item_id")

    # Добавление признаков пользователя
    df_als = df_als.merge(
        user_feats, how="left", on="user_id")

    return df_als[df_als.columns[2:]]

In [ ]:
# Инициализация переменных/констант
BATCH_SIZE = 10_000_000
cb_model = CatBoostClassifier()
cb_model.load_model("../models/model.cb")
ranks = []

# Обработка батчей
for start in range(0, personal_als.shape[0], BATCH_SIZE):
    end = start + BATCH_SIZE
    ranks.extend(cb_model.predict_proba(
        add_features(personal_als.iloc[start:end]))[:, 1])

personal_als["score"] = ranks
personal_als

In [ ]:
# Сортировка рекомендаций
personal_als.sort_values(
    by=["user_id", "score"], 
    ascending=(True, False),
    inplace=True)

# Сохранение датасета
save_data(personal_als, "recommendations.parquet", "recommendations")